In [ ]:
from pymongo import MongoClient
from openai import OpenAI
from transformers import pipeline
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv()

In [53]:
client = MongoClient("192.168.2.70", 27017)
dbs = client.list_database_names()
print(dbs)

['admin', 'config', 'local', 'messages']


In [ ]:
mydb = client["messages"]
mycol = mydb["smsmessages"]

client = OpenAI(api_key = os.getenv("OPENAI_API_KEY"))


In [55]:
classifier = pipeline("zero-shot-classification", device=1)

x = mycol.find()
candidate_labels=["other","travel", "money"]  # changed from finance to money produced MUCH better results
equal = 0
notEqual = 0
unknown = 0
count = 0

for data in x:
    #print(data['message'])
    result = classifier(
        [data['message']],
        candidate_labels=candidate_labels)
    #print("LABEL: ", result[0]['labels'][0], " Score: ", result[0]['scores'][0])
    query_filter = {'message':data['message']}
    
    if(result[0]['scores'][0] >= .50 ):
        if(candidate_labels[data['type']] == result[0]['labels'][0]):
            equal = equal + 1
            #print(candidate_labels[data['type']] ," ", result[0]['labels'][0])
            #print(".")
        else:
            notEqual = notEqual + 1
            #print("NO MATCH: ", candidate_labels[data['type']] ," ", result[0]['labels'][0])
         
        if(result[0]['labels'][0] == "travel"):
            predictValue=1
        elif(result[0]['labels'][0] == "money"):
            predictValue=2
        else:
            predictValue=0
    else:
        if(result[0]['labels'][0] == "other"):
            predictValue = 0
            equal = equal +1
        else:
            predictValue = 9
            unknown = unknown + 1
            
    #print(candidate_labels[data['type']] ," ", result[0]['labels'][0], " predictValue: ", predictValue)
    update_operation = { '$set' : {'predict':predictValue}} #no clear categorization
        
    result = mycol.update_one(query_filter, update_operation)  
    if(count >2000):
        break
    else:
        count = count + 1
    #break
print("Operation Complete - Total: ", unknown + equal + notEqual, " Equals: ", equal, " NotEquals: ", notEqual, " Unknown: ", unknown)


No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1 (https://huggingface.co/facebook/bart-large-mnli).
Using a pipeline without specifying a model name and revision in production is not recommended.


Operation Complete - Total:  1000  Equals:  967  NotEquals:  24  Unknown:  9


In [56]:
myquery =  {"type":0, "predict":0}
tp_Other = mycol.count_documents(myquery)
myquery =  {"type":1, "predict":1}
tp_Travel = mycol.count_documents(myquery)
myquery =  {"type":2, "predict":2}
tp_Finance = mycol.count_documents(myquery)
myquery =  {"type":0}
total_Other = mycol.count_documents(myquery)
myquery =  {"type":1}
total_Travel = mycol.count_documents(myquery)
myquery =  {"type":2}
total_Finance = mycol.count_documents(myquery)

print("Other %: ", "{:.2f}".format(tp_Other/total_Other), " Travel %: ", "{:.2f}".format(tp_Travel/total_Travel), " Finance %: ", "{:.2f}".format(tp_Finance/total_Finance))


Other %:  0.99  Travel %:  0.98  Finance %:  0.61
